Proposition 1: Find customers who are currently active but are considered "low volume", so that they are reached out for re-engagement.

In [ ]:
SELECT
    CustomerID
FROM
    Sales.SalesOrderHeader
WHERE
    YEAR(OrderDate) = 2014
GROUP BY
    CustomerID

EXCEPT

SELECT
    CustomerID
FROM
    Sales.SalesOrderHeader
GROUP BY
    CustomerID
HAVING
    COUNT(SalesOrderID) >= 5;

Proposition 2: Find/display revenue and orders for both North American and European territories to compare their performances to bring a knowledge sharing initiative.

In [ ]:
SELECT
    'North America' AS TerritoryName,
    SUM(TotalDue) AS TotalRevenue,
    COUNT(SalesOrderID) AS TotalOrders
FROM
    Sales.SalesOrderHeader
WHERE
    TerritoryID = 1

UNION ALL

SELECT
    'Europe' AS TerritoryName,
    SUM(TotalDue) AS TotalRevenue,
    COUNT(SalesOrderID) AS TotalOrders
FROM
    Sales.SalesOrderHeader
WHERE
    TerritoryID = 4
ORDER BY
    TotalRevenue DESC;

> Proposition 3: Find the product id and product name in order that currently have a stock a stock quantity greater than zero. This will give us a list of obsolete inventory which will be reviewed for immediate accounting write-off to accurately reflect asset-value.

In [ ]:
SELECT
    p.ProductID
FROM
    Production.ProductInventory AS pi
INNER JOIN
    Production.Product AS p ON pi.ProductID = p.ProductID
GROUP BY
    p.ProductID
EXCEPT
SELECT
    sod.ProductID
FROM
    Sales.SalesOrderDetail AS sod
INNER JOIN
    Sales.SalesOrderHeader AS soh ON sod.SalesOrderID = soh.SalesOrderID
WHERE
    YEAR(soh.OrderDate) = 2014
GROUP BY
    sod.ProductID
ORDER BY
    ProductID;

Proposition 4: Find the total gross profit and total units sold for bikes vs accessories in order to see which of the two primary categories are generating the most profit for the company. That way a company can see where to direct more of their merketing budget, maximizing their profits.

In [ ]:
SELECT
    'Bikes' AS ProductCategory,
    SUM(sod.LineTotal - (sod.OrderQty * p.StandardCost)) AS TotalGrossProfit,
    SUM(sod.OrderQty) AS TotalUnitsSold
FROM
    Sales.SalesOrderDetail AS sod
JOIN
    Production.Product AS p ON sod.ProductID = p.ProductID
JOIN
    Production.ProductSubcategory AS ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
JOIN
    Production.ProductCategory AS pc ON ps.ProductCategoryID = pc.ProductCategoryID
WHERE
    pc.Name = 'Bikes'
GROUP BY
    pc.Name

UNION ALL

SELECT
    'Accessories' AS ProductCategory,
    SUM(sod.LineTotal - (sod.OrderQty * p.StandardCost)) AS TotalGrossProfit,
    SUM(sod.OrderQty) AS TotalUnitsSold
FROM
    Sales.SalesOrderDetail AS sod
JOIN
    Production.Product AS p ON sod.ProductID = p.ProductID
JOIN
    Production.ProductSubcategory AS ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
JOIN
    Production.ProductCategory AS pc ON ps.ProductCategoryID = pc.ProductCategoryID
WHERE
    pc.Name = 'Accessories'
GROUP BY
    pc.Name
ORDER BY
    TotalGrossProfit DESC;

Proposition 5: Find monetary value and total orders for two key suppliers in order to see which of them has higer expidenture to target them for renogiation.

In [ ]:
SELECT
    'Vendor A' AS VendorName,
    SUM(pod.OrderQty * pod.UnitPrice) AS TotalPurchaseValue,
    COUNT(DISTINCT pod.PurchaseOrderID) AS TotalOrders
FROM
    Purchasing.PurchaseOrderDetail AS pod
JOIN
    Purchasing.PurchaseOrderHeader AS poh ON pod.PurchaseOrderID = poh.PurchaseOrderID
WHERE
    poh.VendorID = 1640
UNION ALL

SELECT
    'Vendor B' AS VendorName,
    SUM(pod.OrderQty * pod.UnitPrice) AS TotalPurchaseValue,
    COUNT(DISTINCT pod.PurchaseOrderID) AS TotalOrders
FROM
    Purchasing.PurchaseOrderDetail AS pod
JOIN
    Purchasing.PurchaseOrderHeader AS poh ON pod.PurchaseOrderID = poh.PurchaseOrderID
WHERE
    poh.VendorID = 1650
ORDER BY
    TotalPurchaseValue DESC;

Proposition 6: Display total sales revenue and total order count for both in store and online to see which is the higher performing customer aquistion.

In [ ]:
SELECT
    'Online Sales (Direct)' AS SalesChannel,
    SUM(soh.TotalDue) AS TotalRevenue,
    COUNT(soh.SalesOrderID) AS TotalOrders
FROM
    Sales.SalesOrderHeader AS soh
JOIN
    Sales.Customer AS c ON soh.CustomerID = c.CustomerID
WHERE
    c.StoreID IS NULL

UNION ALL

SELECT
    'Retail/Store Sales' AS SalesChannel,
    SUM(soh.TotalDue) AS TotalRevenue,
    COUNT(soh.SalesOrderID) AS TotalOrders
FROM
    Sales.SalesOrderHeader AS soh
JOIN
    Sales.Customer AS c ON soh.CustomerID = c.CustomerID
WHERE
    c.StoreID IS NOT NULL

ORDER BY
    TotalRevenue DESC;

Proposition 7: Display standard cost of all manufactured products as of today vs a date such as January 1, 2013 in order to seethe overall change in manufacturing costs.

In [ ]:
SELECT
    'Current Standard Cost' AS CostPeriod,
    SUM(p.StandardCost) AS TotalStandardCost
FROM
    Production.Product AS p
WHERE
    p.MakeFlag = 1

UNION ALL

SELECT
    'Cost as of Jan 1, 2013' AS CostPeriod,
    SUM(pc.StandardCost) AS TotalStandardCost
FROM
    Production.ProductCostHistory AS pc
JOIN
    Production.Product AS p ON pc.ProductID = p.ProductID
WHERE
    p.MakeFlag = 1 
    AND pc.StartDate <= '20130101'
    AND (pc.EndDate IS NULL OR pc.EndDate > '20130101')

ORDER BY
    TotalStandardCost DESC;

Proposition 8: Display the total number of work orders and average time to manufacture for road and mountain bikes in order to see which is more efficient to manufacture.

In [ ]:
SELECT
    'Road Bike Line' AS ProductLineName,
    COUNT(wo.WorkOrderID) AS TotalWorkOrders,

    AVG(p.DaysToManufacture) AS AverageDaysToManufacture
FROM
    Production.WorkOrder AS wo
JOIN
    Production.Product AS p ON wo.ProductID = p.ProductID
WHERE
    p.ProductLine = 'R'

UNION ALL

SELECT
    'Mountain Bike Line' AS ProductLineName,
    COUNT(wo.WorkOrderID) AS TotalWorkOrders,
    AVG(p.DaysToManufacture) AS AverageDaysToManufacture
FROM
    Production.WorkOrder AS wo
JOIN
    Production.Product AS p ON wo.ProductID = p.ProductID
WHERE
    p.ProductLine = 'M'

ORDER BY
    AverageDaysToManufacture DESC;

Proposition 9: Display total commision amount and average sales quota for Northwest territory and compare the metrics to that of the Southeast, in order to award an operational bonus to the more profitable with respect to their target.

In [ ]:
SELECT
    'Northwest Territory' AS TerritoryName,
    SUM(sp.SalesYTD * sp.CommissionPct) AS TotalCommissionAmount,
    AVG(sp.SalesQuota) AS AverageSalesQuota
FROM
    Sales.SalesPerson AS sp
JOIN
    Sales.SalesTerritory AS st ON sp.TerritoryID = st.TerritoryID
WHERE
    st.Name = 'Northwest'

UNION ALL

SELECT
    'Southeast Territory' AS TerritoryName,
    SUM(sp.SalesYTD * sp.CommissionPct) AS TotalCommissionAmount,
    AVG(sp.SalesQuota) AS AverageSalesQuota
FROM
    Sales.SalesPerson AS sp
JOIN
    Sales.SalesTerritory AS st ON sp.TerritoryID = st.TerritoryID
WHERE
    st.Name = 'Southeast'

ORDER BY
    TotalCommissionAmount DESC;

Proposition 10: Display the total line items sold and the everage unit price for products sold with a discount and compare with those of undiscounted products to see if discounts are successfully driving higher volume or simply cutting off into margins.

In [ ]:
SELECT 
    'Discounted Sales (Discount > 0)' AS SalesStrategy,
    SUM(sod.OrderQty) AS TotalLineItemsSold,
    AVG(sod.UnitPrice) AS AverageUnitPrice
FROM
    Sales.SalesOrderDetail AS sod
WHERE
    sod.UnitPriceDiscount > 0.00

UNION ALL

SELECT
    'Undiscounted Sales (Discount = 0)' AS SalesStrategy,
    SUM(sod.OrderQty) AS TotalLineItemsSold,
    AVG(sod.UnitPrice) AS AverageUnitPrice
FROM
    Sales.SalesOrderDetail AS sod
WHERE
    sod.UnitPriceDiscount = 0.00

ORDER BY
    TotalLineItemsSold DESC;